In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ImagePipeline-Full") \
    .master("yarn") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

# 读取 HDFS 上的所有图片
all_images = spark.read.format("binaryFile") \
    .load("/user/root/image-pipeline/input/*.jpg")

print(f"总图片数: {all_images.count()}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/07 17:30:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/07 17:30:21 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/07 17:30:22 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/06/07 17:31:10 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/06/07 17:31:25 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/06/07 17:31:40 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered an

总图片数: 500


In [2]:
import cv2
import numpy as np
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType

def compute_sharpness(binary_data):
    nparr = np.frombuffer(binary_data, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()
    return float(variance)

sharpness_udf = udf(compute_sharpness, FloatType())

images_with_sharpness = all_images.withColumn("sharpness", sharpness_udf("content"))
images_with_sharpness.select("path", "sharpness").show(5, truncate=False)

+---------------------------------------------------------------+---------+
|path                                                           |sharpness|
+---------------------------------------------------------------+---------+
|hdfs://master:9000/user/root/image-pipeline/input/cat_10073.jpg|563.53296|
|hdfs://master:9000/user/root/image-pipeline/input/dog_10173.jpg|270.28827|
|hdfs://master:9000/user/root/image-pipeline/input/dog_10058.jpg|119.94306|
|hdfs://master:9000/user/root/image-pipeline/input/dog_1017.jpg |1754.0614|
|hdfs://master:9000/user/root/image-pipeline/input/cat_1002.jpg |134.78423|
+---------------------------------------------------------------+---------+
only showing top 5 rows



In [3]:
images_with_sharpness.describe("sharpness").show()

[Stage 5:=========================================================(16 + 0) / 16]

+-------+-----------------+
|summary|        sharpness|
+-------+-----------------+
|  count|              500|
|   mean|957.2893858804703|
| stddev|1773.055478155212|
|    min|         7.514896|
|    max|        16019.474|
+-------+-----------------+



In [4]:
from pyspark.sql.functions import col

images_with_sharpness.select("path", "sharpness") \
    .orderBy(col("sharpness").asc()) \
    .show(10, truncate=False)

[Stage 8:=====================================================>   (15 + 1) / 16]

+---------------------------------------------------------------+---------+
|path                                                           |sharpness|
+---------------------------------------------------------------+---------+
|hdfs://master:9000/user/root/image-pipeline/input/cat_10183.jpg|7.514896 |
|hdfs://master:9000/user/root/image-pipeline/input/dog_10007.jpg|11.416378|
|hdfs://master:9000/user/root/image-pipeline/input/cat_0.jpg    |14.657422|
|hdfs://master:9000/user/root/image-pipeline/input/dog_10196.jpg|17.405205|
|hdfs://master:9000/user/root/image-pipeline/input/dog_10220.jpg|18.44802 |
|hdfs://master:9000/user/root/image-pipeline/input/cat_10145.jpg|22.720016|
|hdfs://master:9000/user/root/image-pipeline/input/cat_10041.jpg|25.408588|
|hdfs://master:9000/user/root/image-pipeline/input/cat_10023.jpg|28.317905|
|hdfs://master:9000/user/root/image-pipeline/input/dog_10023.jpg|33.201664|
|hdfs://master:9000/user/root/image-pipeline/input/cat_1015.jpg |33.280125|
+-----------

In [5]:
from pyspark.sql.functions import col

SHARPNESS_THRESHOLD = 20

sharp_images = images_with_sharpness.filter(col("sharpness") >= SHARPNESS_THRESHOLD)
blurred_images = images_with_sharpness.filter(col("sharpness") < SHARPNESS_THRESHOLD)

print(f"总图片数: {images_with_sharpness.count()}")
print(f"清晰图片数 (sharpness >= {SHARPNESS_THRESHOLD}): {sharp_images.count()}")
print(f"模糊图片数 (sharpness < {SHARPNESS_THRESHOLD}): {blurred_images.count()}")

总图片数: 500


清晰图片数 (sharpness >= 20): 495


[Stage 15:=============================================>          (13 + 2) / 16]

模糊图片数 (sharpness < 20): 5


In [6]:
import io
import imagehash
from PIL import Image
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# 定义感知哈希 UDF
def compute_phash(binary_data):
    img = Image.open(io.BytesIO(binary_data))
    hash_value = imagehash.phash(img)
    return str(hash_value)

phash_udf = udf(compute_phash, StringType())

# 应用到清晰图片上
sharp_with_hash = sharp_images.withColumn("phash", phash_udf("content"))

# 显示前几行，确认哈希值已计算
sharp_with_hash.select("path", "phash").show(5, truncate=False)

+---------------------------------------------------------------+----------------+
|path                                                           |phash           |
+---------------------------------------------------------------+----------------+
|hdfs://master:9000/user/root/image-pipeline/input/cat_10073.jpg|ec63c71c9a90b993|
|hdfs://master:9000/user/root/image-pipeline/input/dog_10173.jpg|c5603e58d33f486b|
|hdfs://master:9000/user/root/image-pipeline/input/dog_10058.jpg|c13bcce01be473d1|
|hdfs://master:9000/user/root/image-pipeline/input/dog_1017.jpg |d8a62dc5cc3c3971|
|hdfs://master:9000/user/root/image-pipeline/input/cat_1002.jpg |dc6d20c3cd9839e3|
+---------------------------------------------------------------+----------------+
only showing top 5 rows



In [7]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType

# 定义汉明距离 UDF
def hamming_distance(hash1, hash2):
    return imagehash.hex_to_hash(hash1) - imagehash.hex_to_hash(hash2)

hamming_udf = udf(hamming_distance, IntegerType())

# 选取需要的列
image_hashes = sharp_with_hash.select("path", "phash")

# 自连接：找出所有图片对，排除自己和自己，且每对只出现一次
duplicate_pairs = image_hashes.alias("a") \
    .join(image_hashes.alias("b"),
          col("a.path") < col("b.path"),
          "inner") \
    .withColumn("distance", hamming_udf(col("a.phash"), col("b.phash"))) \
    .filter(col("distance") <= 10)

# 显示重复图片对
duplicate_pairs.select(col("a.path").alias("image_1"),
                       col("b.path").alias("image_2"),
                       col("distance")).show(truncate=False)

print(f"重复图片对数: {duplicate_pairs.count()}")

+-------+-------+--------+
|image_1|image_2|distance|
+-------+-------+--------+
+-------+-------+--------+



[Stage 24:=============================================>          (13 + 2) / 16]

重复图片对数: 0


In [8]:
# 选择需要的列，保存为 Parquet 格式
sharp_with_hash.select("path", "sharpness", "phash") \
    .write.mode("overwrite").parquet("/user/root/image-pipeline/output/cleaned_images")

In [9]:
# 读取保存的 Parquet 文件，验证行数
cleaned_df = spark.read.parquet("/user/root/image-pipeline/output/cleaned_images")
print(f"清洗后图片数: {cleaned_df.count()}")

# 查看前几行
cleaned_df.show(5, truncate=False)

清洗后图片数: 495
+---------------------------------------------------------------+---------+----------------+
|path                                                           |sharpness|phash           |
+---------------------------------------------------------------+---------+----------------+
|hdfs://master:9000/user/root/image-pipeline/input/dog_10197.jpg|210.77902|9ee060bc2b3b4717|
|hdfs://master:9000/user/root/image-pipeline/input/dog_10133.jpg|372.73416|cc7133ce8c91b1d9|
|hdfs://master:9000/user/root/image-pipeline/input/cat_1022.jpg |739.3461 |85bcc1530ca5ee5b|
|hdfs://master:9000/user/root/image-pipeline/input/cat_10109.jpg|622.13055|a383582cc6ea39fa|
|hdfs://master:9000/user/root/image-pipeline/input/cat_10211.jpg|347.60986|948d4bd266e366e4|
+---------------------------------------------------------------+---------+----------------+
only showing top 5 rows



In [1]:
spark.stop()

NameError: name 'spark' is not defined